Persiapan

In [2]:
import os
import shutil
import time
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)
from pyspark.sql.functions import col, sum, count, window

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas10_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat.")

SparkSession berhasil dibuat.


A. Menyiapkan Streaming Source (bobot 15%)

Definisikan skema eksplisit (StructType) untuk data pesanan dengan kolom: pesanan_id (string), nama_cabang (string), nama_menu (string), qty (integer), harga_satuan (integer), waktu_pesan (timestamp). Buat folder simulasi streaming baru bernama tugas10_stream, lalu definisikan readStream dari folder tersebut.

In [3]:
# Membuat folder simulasi streaming
folder_stream = "tugas10_stream"

shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# Mendefinisikan skema data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membaca data menggunakan readStream
df_stream = spark.readStream \
    .schema(skema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .csv(folder_stream)

print("Folder streaming berhasil disiapkan:", folder_stream)
df_stream.printSchema()

Folder streaming berhasil disiapkan: tugas10_stream
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



B. Agregasi per Cabang (bobot 20%)

Tambahkan kolom turunan total_harga (qty x harga_satuan) pada df_stream, lalu buat agregasi groupBy("nama_cabang") yang menghitung total total_harga dan jumlah pesanan. Jalankan sebagai streaming query dengan sink memory, queryName="agregasi_cabang".

In [4]:
# Membuat kolom turunan total_harga
df_stream_harga = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan nama cabang
agregasi_cabang = df_stream_harga.groupBy("nama_cabang") \
    .agg(
        sum("total_harga").alias("total_pendapatan"),
        count("*").alias("jumlah_pesanan")
    )

# Menjalankan streaming query
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
+-----------+----------------+--------------+



C. Agregasi Berbasis Window Waktu (bobot 20%)

Buat streaming query kedua (definisikan ulang readStream dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan. Gunakan sink memory, queryName="agregasi_waktu".

In [5]:
# Membaca ulang sumber data streaming
df_window = spark.readStream \
    .schema(skema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .csv(folder_stream)

# Membuat kolom total_harga
df_window_harga = df_window.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Agregasi berdasarkan window waktu 2 menit
agregasi_waktu = df_window_harga.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    sum("total_harga").alias("total_pendapatan")
)

# Menjalankan streaming query kedua
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

spark.sql("SELECT * FROM agregasi_waktu").show(truncate=False)

+------+----------------+
|window|total_pendapatan|
+------+----------------+
+------+----------------+



D. Menguji dengan Data Simulasi (bobot 25%)

Kirim minimal 3 batch data (masing-masing batch berupa satu berkas CSV baru dengan jeda time.sleep() di antaranya) ke folder tugas10_stream, dengan kombinasi nama_cabang yang bervariasi. Setelah setiap batch, tampilkan hasil agregasi_cabang untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [6]:
from datetime import datetime, timedelta

# Data pertama
batch1 = pd.DataFrame([
    ["P001", "Magelang", "Kopi Susu", 2, 18000, "2026-11-11 09:00:10"],
    ["P002", "Magelang", "Americano", 1, 20000, "2026-11-11 09:00:30"],
    ["P003", "Solo", "Cappuccino", 1, 28000, "2026-11-11 09:01:00"]
], columns=[
    "pesanan_id",
    "nama_cabang",
    "nama_menu",
    "qty",
    "harga_satuan",
    "waktu_pesan"
])

batch1.to_csv(
    f"{folder_stream}/batch1.csv",
    index=False
)

time.sleep(5)

query1.processAllAvailable()
query2.processAllAvailable()

spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           28000|             1|
|   Magelang|           56000|             2|
+-----------+----------------+--------------+



In [7]:
# Data kedua
batch2 = pd.DataFrame([
    ["P004", "Yogyakarta", "Matcha Latte", 2, 25000, "2026-11-11 09:02:10"],
    ["P005", "Magelang", "Kopi Hitam", 3, 15000, "2026-11-11 09:02:30"],
    ["P006", "Solo", "Espresso", 1, 20000, "2026-11-11 09:03:00"]
], columns=[
    "pesanan_id",
    "nama_cabang",
    "nama_menu",
    "qty",
    "harga_satuan",
    "waktu_pesan"
])

batch2.to_csv(
    f"{folder_stream}/batch2.csv",
    index=False
)

time.sleep(5)

query1.processAllAvailable()
query2.processAllAvailable()

spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           48000|             2|
| Yogyakarta|           50000|             1|
|   Magelang|          101000|             3|
+-----------+----------------+--------------+



In [8]:
# Data ketiga
batch3 = pd.DataFrame([
    ["P007", "Magelang", "Caramel Latte", 1, 45000, "2026-11-11 09:04:10"],
    ["P008", "Yogyakarta", "Latte", 1, 20000, "2026-11-11 09:04:30"]
], columns=[
    "pesanan_id",
    "nama_cabang",
    "nama_menu",
    "qty",
    "harga_satuan",
    "waktu_pesan"
])

batch3.to_csv(
    f"{folder_stream}/batch3.csv",
    index=False
)

time.sleep(5)

query1.processAllAvailable()
query2.processAllAvailable()

spark.sql("SELECT * FROM agregasi_cabang").show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           48000|             2|
| Yogyakarta|           70000|             2|
|   Magelang|          146000|             4|
+-----------+----------------+--------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [9]:
# 1. Menghentikan seluruh streaming query yang masih aktif
for query in spark.streams.active:
    query.stop()

print("Semua streaming query berhasil dihentikan.")

# 2. Menghapus folder output dan checkpoint lama
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint_baru", ignore_errors=True)

# Membuat folder output
os.makedirs("tugas10_output", exist_ok=True)

print("Folder output dan checkpoint siap digunakan.")

# 3. Membuat streaming query ketiga
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint_baru") \
    .outputMode("append") \
    .start()

print("Streaming query ketiga berhasil dijalankan.")

# 4. Membuat batch data baru
data_batch4 = [
    ("P009", "Magelang", "Kopi Susu", 1, 18000, "2026-11-11 09:05:10")
]

df_batch4 = pd.DataFrame(
    data_batch4,
    columns=[
        "pesanan_id",
        "nama_cabang",
        "nama_menu",
        "qty",
        "harga_satuan",
        "waktu_pesan"
    ]
)

# Menyimpan batch ke folder streaming
df_batch4.to_csv(
    "tugas10_stream/batch4.csv",
    index=False
)

print("Batch 4 berhasil dibuat.")

# 5. Menunggu dan memproses data streaming
time.sleep(5)
query3.processAllAvailable()

print("Data berhasil diproses.")

# 6. Memeriksa file Parquet yang dihasilkan
print("\nDaftar file dalam folder output:")
print(os.listdir("tugas10_output"))

# 7. Membaca kembali data Parquet
df_hasil = spark.read.parquet("tugas10_output")

print("\nHasil pembacaan data Parquet:")
df_hasil.show(truncate=False)

# 8. Menghentikan streaming query
query3.stop()

print("Streaming query ketiga berhasil dihentikan.")

Semua streaming query berhasil dihentikan.
Folder output dan checkpoint siap digunakan.
Streaming query ketiga berhasil dijalankan.
Batch 4 berhasil dibuat.
Data berhasil diproses.

Daftar file dalam folder output:
['part-00003-638594e7-caed-482a-881d-6558e6fd43fe-c000.snappy.parquet', '.part-00003-638594e7-caed-482a-881d-6558e6fd43fe-c000.snappy.parquet.crc', '.part-00001-7707a774-ec02-4daa-acd2-d613127a462f-c000.snappy.parquet.crc', '_spark_metadata', '.part-00002-45f68d8e-8168-45a5-b697-3ea9890ffaf0-c000.snappy.parquet.crc', 'part-00001-7707a774-ec02-4daa-acd2-d613127a462f-c000.snappy.parquet', 'part-00000-86e45f32-8345-42a9-bfac-3ffebcc4896d-c000.snappy.parquet', 'part-00002-45f68d8e-8168-45a5-b697-3ea9890ffaf0-c000.snappy.parquet', '.part-00000-86e45f32-8345-42a9-bfac-3ffebcc4896d-c000.snappy.parquet.crc']

Hasil pembacaan data Parquet:
+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu    |qty|harga_satuan|waktu_pesan    

In [ ]:
batch4 = pd.DataFrame([
    ["P009", "Magelang", "Kopi Susu", 1, 18000, "2026-11-11 09:05:10"]
], columns=[
    "pesanan_id",
    "nama_cabang",
    "nama_menu",
    "qty",
    "harga_satuan",
    "waktu_pesan"
])

batch4.to_csv(
    f"{folder_stream}/batch4.csv",
    index=False
)

time.sleep(5)

query3.processAllAvailable()

In [61]:
print("Query aktif:", spark.streams.active)

if 'query3' in globals():
    print("Status query3:", query3.status)
    print("Exception query3:", query3.exception())

Query aktif: [<pyspark.sql.streaming.query.StreamingQuery object at 0x795a08897e50>]
Status query3: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}
Exception query3: None


Menghentikan Seluruh Streaming Query

In [10]:
query1.stop()
query2.stop()
query3.stop()

spark.stop()

print("Seluruh streaming query berhasil dihentikan.")
print("SparkSession ditutup.")

Seluruh streaming query berhasil dihentikan.
SparkSession ditutup.
